# Setup

In [ ]:
!pip install torch==1.13.1

In [ ]:
!git clone https://github.com/facebookresearch/colorlessgreenRNNs

%cd /content/colorlessgreenRNNs/src

!wget https://dl.fbaipublicfiles.com/colorless-green-rnns/best-models/English/hidden650_batch128_dropout0.2_lr20.0.pt
!wget -P ../data/lm/English/ https://dl.fbaipublicfiles.com/colorless-green-rnns/training-data/English/train.txt
!wget -P ../data/lm/English/ https://dl.fbaipublicfiles.com/colorless-green-rnns/training-data/English/test.txt
!wget -P ../data/lm/English/ https://dl.fbaipublicfiles.com/colorless-green-rnns/training-data/English/valid.txt
!wget -P ../data/lm/English/ https://dl.fbaipublicfiles.com/colorless-green-rnns/training-data/English/vocab.txt

In [49]:
import csv
import pandas as pd
import torch.nn.functional as F
import matplotlib
import seaborn as sns
from tqdm import tqdm

In [ ]:
%cd /content/colorlessgreenRNNs/src/language_models

import torch
import torch.nn as nn
import numpy as np
from model import RNNModel


torch.manual_seed(50360)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(50360)


model_ = None
fn = "../hidden650_batch128_dropout0.2_lr20.0.pt"
with open(fn, "rb") as model_f:
    model_ = torch.load(fn)

# Makes sure your model is loaded onto the GPU (should return true)
next(model_.parameters()).is_cuda

# Construct a new RNNModel using PyTorch 1.x implementations of NN modules
model = RNNModel("LSTM", 50001, 650, 650, 2, 0.2, False)
# Copy over the trained weights from the model loaded in
model.load_state_dict(model_.state_dict())

model = model.cuda()
model.eval()

And set up the `dictionary corpus`


In [5]:
import dictionary_corpus

data_path = "../../data/lm/English"

dictionary = dictionary_corpus.Dictionary(data_path)

print("Vocab size is ", len(dictionary))

Vocab size is  50001


In [6]:
dictionary.word2idx["the"], dictionary.word2idx["."]

(3, 18)

In [48]:
dictionary.word2idx["<unk>"]

62

# Helper Utils

In [16]:
Verbs = {}
with open('../../../verblist_DT_usf_freq.csv', mode ='r')as f:
    csvFile = csv.reader(f, delimiter =';')
    for line in csvFile:
       if not line[0] == "\ufeffV":
          Verbs[line[0].upper()] = {'pres': line[0],
                                    'past': line[-3],
                                    '3sg': line[-2],
                                    'participle': line[-1],
                                    'prep': line[-4]}
Verbs_inv = {}
for verb in Verbs:
    for form in list(Verbs[verb].values())[:4]:
        Verbs_inv[form] = verb

In [52]:
def get_prob_unprimed(sen):
  # given a raw input sentence, convert into token ids on cuda
  indices = [dictionary.word2idx[w] if w in dictionary.word2idx
                                    else dictionary.word2idx["<unk>"]
            for w in sen.capitalize().split()]
  indices = torch.tensor(indices, dtype=torch.long).cuda()

  # hidden has shape [num_layers, batch_size, hidden_dimension]
  # output has shape [seq_len, batch_size, d_vocab]
  output, hidden = model(indices.view(-1, 1), model.init_hidden(1))

  # Get the probability distribution and the log_prob of the corresponding tokens
  probs = F.log_softmax(output.squeeze()[:-1], dim=1)
  probs = probs[range(probs.size(0)), indices[1:]]

  # compute log_probability and perplexity
  sen_len = len(indices)-1
  sen_prob = torch.sum(probs)
  perplexity = torch.exp(sen_prob) ** (-1 / sen_len)

  return sen_prob, perplexity

In [108]:
def get_prob_primed(sen):
    prime = sen.split('.')[0].split()
    prime.append('.')
    prime_len = len(prime)
    target = sen.split('.')[1].lstrip().split()
    target.append('.')
    prime.extend(target)

    indices = [dictionary.word2idx[w] if w in dictionary.word2idx
                                        else dictionary.word2idx["<unk>"]
                for w in prime]
    indices = torch.tensor(indices, dtype=torch.long).cuda()
    output, hidden = model(indices.view(-1, 1), model.init_hidden(1))

    probs = F.log_softmax(output.squeeze()[prime_len:-1], dim=1)
    probs = probs[range(probs.size(0)), indices[prime_len+1:]]

    sen_len = len(target)-1
    sen_prob = torch.sum(probs)
    perplexity = torch.exp(sen_prob) ** (-1 / sen_len)

    return sen_prob.item(), perplexity.item()

# Running on CORE_dative_pronoun

In [10]:
with open('../../../CORE_dative_pronouns.csv') as file:
    lines = list(csv.reader(file, delimiter=','))[1:]

all = []
for line in lines:
  for sen in line:
    all.extend(sen.split(' '))

# converting a list of string tokens to indices
indices = torch.tensor([dictionary.word2idx[w] if w in dictionary.word2idx
                                  else dictionary.word2idx["<unk>"]
            for w in list(set(all))], dtype=torch.long).cuda()
if any(i==dictionary.word2idx["<unk>"] for i in indices):
  print('bad!')

In [17]:
with open('../../../CORE_dative_pronouns.csv') as file:
    lines = list(csv.reader(file, delimiter=','))[1:]

DO = [line[1] for line in lines[1:]]
DO.extend([line[3] for line in lines[1:]])
DO = list(set(DO))
PD = [line[0] for line in lines[1:]]
PD.extend([line[2] for line in lines[1:]])
PD = list(set(PD))

lst = []
for sentence in DO:
    lst.append([sentence, 'DO', Verbs_inv[sentence.split(' ')[2]]])
for sentence in PD:
    lst.append([sentence, 'PD', Verbs_inv[sentence.split(' ')[2]]])

NoPriming = pd.DataFrame(lst, columns =['text', 'structure', 'verb'])
NoPriming

,text,structure,verb
0,a secretary sold her a plate .,DO,SELL
1,the buddy supplied her the cup .,DO,SUPPLY
2,a sheriff left it a knife .,DO,LEAVE
3,the brother sold us the pie .,DO,SELL
4,an employer fed her a cheese .,DO,FEED
...,...,...,...
43497,the writer threw the pie to me .,PD,THROW
43498,a secretary supplied a coffee to him .,PD,SUPPLY
43499,an author showed a meal to you .,PD,SHOW
43500,a mother made a telephone for them .,PD,MAKE


In [51]:
NoPriming = pd.DataFrame(lst, columns =['text', 'structure', 'verb'])

sen_prob = []
perplexity = []

for sen in tqdm(NoPriming['text'].to_list()):
    prob, perp = get_prob_unprimed(sen)
    sen_prob.append(prob.item())
    perplexity.append(perp.item())

NoPriming['sen_prob'] = sen_prob
NoPriming['perplexity'] = perplexity

NoPriming.to_csv(f'LSTM_NoPriming_pronouns.csv', index=False)

100%|██████████| 43502/43502 [01:54<00:00, 378.73it/s]


# Running on CORE_dative

In [54]:
# check if all tokens are tokenizable!
with open('../../../CORE_dative.csv') as file:
    lines = list(csv.reader(file, delimiter=','))[1:]

all = []
for line in lines:
  for sen in line:
    all.extend(sen.split(' '))

# converting a list of string tokens to indices
indices = torch.tensor([dictionary.word2idx[w] if w in dictionary.word2idx
                                  else dictionary.word2idx["<unk>"]
            for w in list(set(all))], dtype=torch.long).cuda()
if any(i==dictionary.word2idx["<unk>"] for i in indices):
  print('bad!')

In [55]:
with open('../../../CORE_dative.csv') as file:
    lines = list(csv.reader(file, delimiter=','))[1:]

DO = [line[1] for line in lines[1:]]
DO.extend([line[3] for line in lines[1:]])
DO = list(set(DO))
PD = [line[0] for line in lines[1:]]
PD.extend([line[2] for line in lines[1:]])
PD = list(set(PD))

lst = []
for sentence in DO:
    lst.append([sentence, 'DO', Verbs_inv[sentence.split(' ')[2]]])
for sentence in PD:
    lst.append([sentence, 'PD', Verbs_inv[sentence.split(' ')[2]]])

NoPriming = pd.DataFrame(lst, columns =['text', 'structure', 'verb'])
NoPriming

,text,structure,verb
0,a princess brought a corporation an instrument .,DO,BRING
1,a buddy left an uncle a flower .,DO,LEAVE
2,the student purchased the club the coffee .,DO,PURCHASE
3,the friend found the woman the juice .,DO,FIND
4,a king took a parent a key .,DO,TAKE
...,...,...,...
32825,a brother drew a book for a business .,PD,DRAW
32826,the child kept the gun for the secretary .,PD,KEEP
32827,the pilot took the gun to the king .,PD,TAKE
32828,a wife gave a tea to a business .,PD,GIVE


In [56]:
sen_prob = []
perplexity = []

for sen in tqdm(NoPriming['text'].to_list()):
    prob, perp = get_prob_unprimed(sen)
    sen_prob.append(prob.item())
    perplexity.append(perp.item())

NoPriming['sen_prob'] = sen_prob
NoPriming['perplexity'] = perplexity

NoPriming.to_csv(f'LSTM_NoPriming.csv', index=False)

100%|██████████| 32830/32830 [01:21<00:00, 400.68it/s]


# Running with Priming: CORE_dative_pronouns

In [109]:
# ppo, pdo, tpo, tdo
with open('../../../CORE_dative_pronouns.csv') as file:
    lines = list(csv.reader(file, delimiter=','))[1:]

col_names = ['prime_sentence', 'prime_structure', 'prime_verb',
             'target_sentence', 'target_structure', 'target_verb',
             'perplexity', 'log_prob']


rec = []
for line in tqdm(lines):
    # get Prime
    prime_PD = line[0]
    prime_DO = line[1]
    prime_verb = Verbs_inv[line[0].split(' ')[2]]

    # get Target
    target_PD = line[2]
    target_DO = line[3]
    target_verb = Verbs_inv[line[2].split(' ')[2]]

    # PD-PD
    prime_text = prime_PD.capitalize()[:-2]+'.'
    target_text = target_PD.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_PD, 'PD', prime_verb, target_PD, 'PD', target_verb, perplexity, prob])

    # DO-PD
    prime_text = prime_DO.capitalize()[:-2]+'.'
    target_text = target_PD.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_DO, 'DO', prime_verb, target_PD, 'PD', target_verb, perplexity, prob])

    # PD-DO
    prime_text = prime_PD.capitalize()[:-2]+'.'
    target_text = target_DO.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_PD, 'PD', prime_verb, target_DO, 'DO', target_verb, perplexity, prob])

    # DO-DO
    prime_text = prime_DO.capitalize()[:-2]+'.'
    target_text = target_DO.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_DO, 'DO', prime_verb, target_DO, 'DO', target_verb, perplexity, prob])


Parsed = pd.DataFrame(rec, columns =col_names)
Parsed.to_csv(f'LSTM_Priming_pronouns.csv', index=False)


100%|██████████| 15000/15000 [02:46<00:00, 90.20it/s]


# Run with Priming: no pronouns

In [110]:
# ppo, pdo, tpo, tdo
with open('../../../CORE_dative.csv') as file:
    lines = list(csv.reader(file, delimiter=','))[1:]

col_names = ['prime_sentence', 'prime_structure', 'prime_verb',
             'target_sentence', 'target_structure', 'target_verb',
             'perplexity', 'log_prob']


rec = []
for line in tqdm(lines):
    # get Prime
    prime_PD = line[0]
    prime_DO = line[1]
    prime_verb = Verbs_inv[line[0].split(' ')[2]]

    # get Target
    target_PD = line[2]
    target_DO = line[3]
    target_verb = Verbs_inv[line[2].split(' ')[2]]

    # PD-PD
    prime_text = prime_PD.capitalize()[:-2]+'.'
    target_text = target_PD.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_PD, 'PD', prime_verb, target_PD, 'PD', target_verb, perplexity, prob])

    # DO-PD
    prime_text = prime_DO.capitalize()[:-2]+'.'
    target_text = target_PD.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_DO, 'DO', prime_verb, target_PD, 'PD', target_verb, perplexity, prob])

    # PD-DO
    prime_text = prime_PD.capitalize()[:-2]+'.'
    target_text = target_DO.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_PD, 'PD', prime_verb, target_DO, 'DO', target_verb, perplexity, prob])

    # DO-DO
    prime_text = prime_DO.capitalize()[:-2]+'.'
    target_text = target_DO.capitalize()[:-2]+'.'
    text = prime_text + ' ' + target_text
    prob, perplexity = get_prob_primed(text)
    rec.append([prime_DO, 'DO', prime_verb, target_DO, 'DO', target_verb, perplexity, prob])


Parsed = pd.DataFrame(rec, columns =col_names)
Parsed.to_csv(f'LSTM_Priming.csv', index=False)


100%|██████████| 15000/15000 [02:52<00:00, 87.03it/s]
